In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
eurepoc = pd.read_csv('../data/eurepoc_global_dataset_1_3_EDITED.csv')

In [3]:
# we remove unnecessary columns
eurepoc = eurepoc.drop(columns=[
    'number_attributions',
    'number_political_responses',
    'number_legal_responses',
    'receiver_country_alpha_2_code',
    'attribution_source_url',
    'initiator_alpha_2',
    'initiator_country.1',
    'initiator_category.1',
    'source_url',
    'attribution_legal_reference',
    'attribution_legal_reference_subcode',
    'added_to_db',
    'updated_at'
])

In [4]:
def parse_list_with_parentheticals(cell):
    items = []
    last = 0
    depth = 0
    for i, c in enumerate(cell):
        if c == '(':
            depth += 1
        if c == ')':
            depth -= 1
        if c == ';' and depth <= 0:
            items.append(cell[last:i])
            depth = 0
            last = i + 1
    if cell[last:] != '':
        items.append(cell[last:])
    return items

# these columns contain semicolons but do not represent multiple entries per cell
ignored_columns = ['name', 'description', 'disruption', 'physical_effects_temporal']

# these columns are semicolon-delimited, but also occasionally contain semicolons in parentheticals
escaped_columns = ['receiver_name', 'initiator_name', 'attributing_actor', 'political_response_responding_actor']

for series_name, series in eurepoc.items():
    # if we trust column to be singleton (logan reference), we can ignore it
    if series_name in ignored_columns: continue
    # if column not of type string, proceed
    if series.dtype != 'str': continue

    if series_name in escaped_columns:
        # use a complicated and expensive operation when necessary
        lists = series.apply(lambda cell: parse_list_with_parentheticals(str(cell)))
    else:
        # otherwise a simple split works fine
        lists = series.apply(lambda cell: str.split(str(cell), ';'))
        
    # if series does not any list-formatted values, proceed
    if not lists.apply(lambda items: len(items) != 1).any(): continue
    
    eurepoc[series_name] = lists

In [5]:
column_groups = [
    ['inclusion_criterion', 'inclusion_criterion_subcode'],
    ['source_disclosure'],
    ['incident_type'],
    ['receiver_name', 'receiver_country', 'receiver_regions', 'receiver_category', 'receiver_subcategory'],
    ['initiator_name', 'initiator_country', 'initiator_category', 'initiator_subcategory', 'attribution_id', 'attribution_date', 'attribution_type', 'attribution_basis', 'attributing_actor', 'attributing_company', 'attributing_country', 'settled_initiator'],
    ['cyber_conflict_issue'],
    ['offline_conflict_issue', 'offline_conflict_name', 'offline_conflict_intensity', 'offline_conflict_intensity_subcode'],
    ['political_response_date', 'political_response_type', 'political_response_subtype', 'political_response_responding_country', 'political_response_responding_actor'],
    ['il_breach_indicator', 'il_breach_indicator_subcode'],
    ['legal_response_date', 'legal_response_type', 'legal_response_subtype', 'legal_response_responding_country', 'legal_response_responding_actor'],
]

In [6]:
def explode_group(columns):
    # select id and relevant columns, then explode on those columns
    df = eurepoc[['incident_id', *columns]].explode(columns, ignore_index=True)
    eurepoc.drop(columns=columns, inplace=True)
    return df

inclusion = explode_group(column_groups[0])
disclosure = explode_group(column_groups[1])
incident_type = explode_group(column_groups[2])
receiver = explode_group(column_groups[3])
initiator = explode_group(column_groups[4])
cyber_issue = explode_group(column_groups[5])
offline_issue = explode_group(column_groups[6])
political_response = explode_group(column_groups[7])
international_law = explode_group(column_groups[8])
legal_response = explode_group(column_groups[9])

In [7]:
replace_not_available = lambda cell: None if cell == 'Not available' else cell
replace_not_attributed = lambda cell: None if cell == 'Not attributed' else cell

parse_dd_mm_yyyy = lambda cell: None if cell == 'Not available' else pd.to_datetime(cell, dayfirst=True)

In [8]:
inclusion = inclusion.rename(columns={
    'incident_id': 'Incident',
    'inclusion_criterion': 'Criterion',
    'inclusion_criterion_subcode': 'Subcode',
})

inclusion.to_csv('../data/inclusion.csv')

In [9]:
disclosure = disclosure.rename(columns={
    'incident_id': 'Incident',
    'source_disclosure': 'Source'
})

disclosure.to_csv('../data/disclosure.csv')

In [10]:
incident_type = incident_type.rename(columns={
    'incident_id': 'Incident',
    'incident_type': 'Type'
})

incident_type.to_csv('../data/incident_type.csv')

The receiver table requires some special handling since regions are *also* stored as packed lists. Thankfully, this is much easier than the table as a whole since there's only a single level.

In [11]:
# parse regions into list
regions = receiver['receiver_regions'].apply(lambda cell: [region[1:-1] for region in str.split(cell[1:-1], ', ')])
# find the highest number of regions in a single cell
columns = regions.apply(lambda cell: len(cell)).max()
labels = ['Region ' + str(i + 1) for i in range(columns)]
# pad with None so we can zip into a our eventual series
regions = regions.apply(lambda cell: cell + [None] * (columns - len(cell)))
series = zip(*list(regions))

# set each column on receiver
for label, values in zip(labels, series):
    receiver[label] = values

# remove original
receiver = receiver.drop(columns=['receiver_regions'])

In [12]:
receiver = receiver.rename(columns={
    'incident_id': 'Incident',
    'receiver_name': 'Name',
    'receiver_country': 'Country',
    'receiver_category': 'Category',
    'receiver_subcategory': 'Subcategory'
})

receiver = receiver.map(replace_not_available)

receiver.to_csv('../data/receiver.csv')

In [13]:
initiator = initiator.rename(columns={
    'incident_id': 'Incident',
    'initiator_name': 'Name',
    'initiator_country': 'Country',
    'initiator_category': 'Category',
    'initiator_subcategory': 'Subcategory',
    'attribution_id': 'Attribution ID',
    'attribution_date': 'Attribution date',
    'attribution_type': 'Attribution type',
    'attribution_basis': 'Attribution basis',
    'attributing_actor': 'Attribution actor',
    'attributing_company': 'Attribution company',
    'attributing_country': 'Attribution country',
    'settled_initiator': 'Settled with initiator',
})

initiator[['Name', 'Country', 'Category']] = initiator[['Name', 'Country', 'Category']].map(replace_not_attributed)
initiator = initiator.map(replace_not_available)

initiator['Attribution date'] = initiator['Attribution date'].apply(parse_dd_mm_yyyy)
initiator['Settled with initiator'] = initiator['Settled with initiator'].apply(bool)

initiator.to_csv('../data/initiator.csv')

In [14]:
cyber_issue = cyber_issue.rename(columns={
    'incident_id': 'Incident',
    'cyber_conflict_issue': 'Issue'
})

cyber_issue.to_csv('../data/cyber_issue.csv')

In [15]:
offline_issue = offline_issue.rename(columns={
    'incident_id': 'Incident',
    'offline_conflict_issue': 'Issue',
    'offline_conflict_name': 'Name',
    'offline_conflict_intensity': 'Intensity',
    'offline_conflict_intensity_subcode': 'Intensity Subcode'
})

offline_issue = offline_issue.map(replace_not_available)

offline_issue.to_csv('../data/offline_issue.csv')

In [16]:
political_response = political_response.rename(columns={
    'incident_id': 'Incident',
    'political_response_date': 'Date',
    'political_response_type': 'Type',
    'political_response_subtype': 'Subtype',
    'political_response_responding_country': 'Country',
    'political_response_responding_actor': 'Actor'
})

political_response = political_response.map(replace_not_available)
political_response['Date'] = political_response['Date'].apply(parse_dd_mm_yyyy)

political_response.to_csv('../data/political_response.csv')

In [17]:
international_law = international_law.rename(columns={
    'incident_id': 'Incident',
    'il_breach_indicator': 'Indicator',
    'il_breach_indicator_subcode': 'Subcode'
})

international_law.to_csv('../data/international_law.csv')

In [18]:
legal_response = legal_response.rename(columns={
    'incident_id': 'Incident',
    'legal_response_date': 'Date',
    'legal_response_type': 'Type',
    'legal_response_subtype': 'Subtype',
    'legal_response_responding_country': 'Country',
    'legal_response_responding_actor': 'Actor'
})

legal_response['Date'] = legal_response['Date'].apply(lambda date: pd.to_datetime(date) if date != 'Not available' else None)
legal_response[['Type', 'Subtype', 'Country', 'Actor']] = legal_response[['Type', 'Subtype', 'Country', 'Actor']].map(lambda value: None if value == 'Not available' else value)

legal_response.to_csv('../data/legal_response.csv')

In [19]:
eurepoc = eurepoc.rename(columns={
    'incident_id': 'Incident id',
    'name': 'Name',
    'description': 'Description',
    'start_date': 'Start date',
    'end_date': 'End date',
    'zero_days': 'Zero days',
    'zero_days_subcode': 'Zero days subcode',
    'mitre_initial_access': 'MITRE initial access',
    'mitre_impact': 'MITRE impact',
    'user_interaction': 'User interaction',
    'has_disruption': 'Has disruption',
    'data_theft': 'Data theft',
    'disruption': 'Disruption',
    'hijacking': 'Hijacking',
    'physical_effects_spatial': 'Spatial-physical effects',
    'physical_effects_temporal': 'Temporal-physical effects',
    'unweighted_intensity': 'Unweighted intensity',
    'target_multiplier': 'Target multiplier',
    'weighted_intensity': 'Weighted intensity',
    'impact_indicator_label': 'Impact label',
    'impact_indicator_score': 'Impact score',
    'functional_impact': 'Functional impact',
    'intelligence_impact': 'Intelligence impact',
    'affected_entities': 'Affected entities',
    'affected_entities_value': 'Affected entities value',
    'affected_third_countries': 'Affected third countries',
    'affected_third_countries_value': 'Affected third countries value',
    'economic_impact': 'Economic impact',
    'economic_impact_value': 'Economic impact value',
    'economic_impact_currency': 'Economic impact currency',
    'state_responsibility_actor': 'State responsibility actor',
    'evidence_for_sanctions_indicator': 'Evidence for sanctions indicator',
    'response_indicator': 'Response indicator',
    'casualties': 'Casualties'
})

eurepoc = eurepoc.map(replace_not_available)

eurepoc['Start date'] = eurepoc['Start date'].apply(parse_dd_mm_yyyy)
eurepoc['End date'] = eurepoc['End date'].apply(parse_dd_mm_yyyy)

In [20]:
eurepoc['Year'] = eurepoc['Start date'].apply(lambda date: date.year)
eurepoc = eurepoc[(2017 <= eurepoc['Year']) & (eurepoc['Year'] <= 2023)]

pre_2020_condition = eurepoc['Year'] < 2020

pre_2020 = eurepoc[pre_2020_condition]
post_2020 = eurepoc[~pre_2020_condition]

pre_2020.to_csv('../data/pre_2020.csv')
post_2020.to_csv('../data/post_2020.csv')